# Daily Citi Bike demand: linear regression

Predict **daily trip starts per station** from the combined monthly gold stores. Features and labels join one-to-one on `station_id` and `snapshot_date`.

| Period | Role |
|---|---|
| 2016 | Train |
| 2017 Q1 | Tune with randomized search |
| 2017 Q2 | Test |
| 2017 Q3 | Out-of-time 1 |
| 2017 Q4 | Out-of-time 2 |

**Selection metric: mean absolute error (MAE), measured in trips per station per day.** Lower is better. Splits follow dates, so every station on a date belongs to the same split.

Ordinary least squares is the baseline. We tune **Ridge regression**, a linear regression model with L2 regularization, because ordinary least squares has few meaningful tuning parameters. Only 2016 fits the final models; 2017 Q1 selects Ridge settings. Test and out-of-time data do not participate in fitting or selection.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import loguniform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
N_ITER = 20
pd.set_option("display.max_columns", 30)

# Works when launched from the repository root or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "datamart_monthly/gold/combined/gold_feature_store.parquet").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run scripts/combine_gold_stores.py from the repository root first.")
COMBINED = ROOT / "datamart_monthly/gold/combined"

## Load and validate the combined stores

Reject duplicate or unmatched station/date keys instead of silently dropping rows. The target is `label`; `label_def` and the raw snapshot date are excluded from the predictors.

In [ ]:
features = pd.read_parquet(COMBINED / "gold_feature_store.parquet")
if any(c.endswith("_7d") for c in features.columns):
    raise ValueError("Rebuild gold features and combined exports: expected same-day weather, found 7-day aggregates.")
labels = pd.read_parquet(COMBINED / "gold_label_store.parquet")
KEYS = ["station_id", "snapshot_date"]
for name, frame in [("features", features), ("labels", labels)]:
    frame["snapshot_date"] = pd.to_datetime(frame["snapshot_date"])
    if frame[KEYS].isna().any().any() or frame.duplicated(KEYS).any():
        raise ValueError(f"Invalid or duplicate station/date keys in {name}")
if set(labels["label_def"].dropna().unique()) != {"trip_starts_1d"}:
    raise ValueError("Expected daily trip-start labels")

data = features.merge(labels, on=KEYS, how="outer", validate="one_to_one", indicator=True)
if not data["_merge"].eq("both").all():
    raise ValueError(f"Unmatched feature/label keys: {data['_merge'].value_counts().to_dict()}")
data = data.drop(columns="_merge").sort_values(["snapshot_date", "station_id"]).reset_index(drop=True)
if data["label"].isna().any() or data["label"].lt(0).any():
    raise ValueError("Labels must be non-null and non-negative")
print(f"Joined {len(data):,} station/day rows across {data.station_id.nunique():,} stations")
print(f"Dates: {data.snapshot_date.min().date()} to {data.snapshot_date.max().date()}")
display(data.head())

## Split chronologically

No random row split is used. A random row split would mix earlier and later dates. The first snapshot can be empty because station activity requires previous trip history.

**Availability assumption:** weather summarizes observed hourly measurements on the **same snapshot date**, joined by `snapshot_date`. These observations are known after that day, so the weather variant is a historical analysis / observed-weather benchmark. It does not represent a deployable day-ahead forecast. The no-weather variant retains trip history through D-1 and calendar features.

In [ ]:
periods = {
    "Train 2016": ("2016-01-01", "2017-01-01"),
    "Tune 2017 Q1": ("2017-01-01", "2017-04-01"),
    "Test 2017 Q2": ("2017-04-01", "2017-07-01"),
    "OOT 1: 2017 Q3": ("2017-07-01", "2017-10-01"),
    "OOT 2: 2017 Q4": ("2017-10-01", "2018-01-01"),
}
splits = {name: data.loc[(data.snapshot_date >= start) & (data.snapshot_date < end)].copy()
          for name, (start, end) in periods.items()}
if sum(map(len, splits.values())) != len(data):
    raise ValueError("The combined data contains dates outside the configured split periods")
if any(frame.empty for frame in splits.values()):
    raise ValueError("Every split must contain rows; check the combined input date coverage")
summary = pd.DataFrame([{
    "split": name, "rows": len(frame), "stations": frame.station_id.nunique(),
    "first_date": frame.snapshot_date.min().date(), "last_date": frame.snapshot_date.max().date(),
    "mean_daily_station_trips": frame.label.mean(),
} for name, frame in splits.items()]).set_index("split")
display(summary)
train = splits["Train 2016"]
tune = splits["Tune 2017 Q1"]

## Preprocess inside each model pipeline

Standardize numeric features and impute missing numeric values using training medians. One-hot encode station, weekday and month so their numeric labels do not imply linear ordering. Unknown stations are allowed. Preprocessing is fitted only on the training fold, including during search.

Federal holiday features come from `data/us_federal_holidays_2016_2017.csv`. `fe_is_holiday` flags the actual holiday date; `fe_is_observed_holiday` flags its federal observed date, which can differ for weekends. Both are numeric binary calendar features, retained in the no-weather models. Holiday dates are known in advance.

`fe_is_long_weekend` flags every day in a run of at least three consecutive weekend/holiday days. It uses actual and observed dates from the supplied calendar, without assuming bridge days (for example, the Friday after Thanksgiving). Holiday entries in adjacent years are needed for complete boundary coverage; the current source does not include New Year's Day 2018, so December 30–31, 2017 are not classified as a long weekend.

In [ ]:
feature_columns = [c for c in features.columns if c != "snapshot_date"]
categorical_columns = ["station_id", "fe_day_of_week", "fe_month"]
numeric_columns = [c for c in feature_columns if c not in categorical_columns]
if {"label", "label_def"} & set(feature_columns):
    raise ValueError("Target leakage in feature columns")

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scaler", StandardScaler()),
    ]), numeric_columns),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=True), categorical_columns),
], sparse_threshold=1.0)

def make_model(regressor):
    return Pipeline([("preprocess", clone(preprocessor)), ("model", regressor)])

X_train, y_train = train[feature_columns], train["label"]
X_tune, y_tune = tune[feature_columns], tune["label"]
print("Numeric:", numeric_columns)
print("Categorical:", categorical_columns)

## Randomized tuning on the fixed Q1 validation period

`PredefinedSplit` marks 2016 rows as training (`-1`) and 2017 Q1 rows as validation (`0`). Each candidate sees exactly this chronological split. Search maximizes negative MAE, which is equivalent to minimizing MAE.

We search 20 combinations of Ridge's regularization strength and intercept setting. `refit=False` prevents automatic training on the combined training and tuning data; the selected model is explicitly fitted on 2016 below.

In [ ]:
X_search = pd.concat([X_train, X_tune], ignore_index=True)
y_search = pd.concat([y_train, y_tune], ignore_index=True)
validation_fold = np.r_[np.full(len(train), -1), np.zeros(len(tune), dtype=int)]
search = RandomizedSearchCV(
    estimator=make_model(Ridge(solver="lsqr", tol=1e-4)),
    param_distributions={
        "model__alpha": loguniform(1e-3, 1e4),
        "model__fit_intercept": [True, False],
    },
    n_iter=N_ITER,
    scoring="neg_mean_absolute_error",
    cv=PredefinedSplit(validation_fold),
    refit=False,
    n_jobs=1,
    random_state=RANDOM_STATE,
    error_score="raise",
)
search.fit(X_search, y_search)
print("Best parameters:", search.best_params_)
print(f"Best tuning MAE: {-search.best_score_:.4f}")
search_results = pd.DataFrame(search.cv_results_)
search_results["tune_mae"] = -search_results["mean_test_score"]
display(search_results.sort_values("rank_test_score")[[
    "rank_test_score", "param_model__alpha", "param_model__fit_intercept", "tune_mae", "mean_fit_time"
]].head(10))

## Fit baseline and selected model on 2016

Predictions remain raw linear-regression outputs, including possible negative values. Search and reported scores use the same prediction rule. No test or out-of-time labels are used to change parameters or predictions.

In [ ]:
ordinary_model = make_model(LinearRegression())
ordinary_model.fit(X_train, y_train)
best_model = clone(search.estimator).set_params(**search.best_params_)
best_model.fit(X_train, y_train)
print("Fitted ordinary least squares and tuned Ridge using 2016 only.")

## Which features matter to the tuned Ridge model?

Permutation importance measures **how much MAE increases when a feature is shuffled**, holding the fitted model fixed. A larger positive increase means the model relies more on that feature. Near-zero or negative importance means shuffling did not reliably hurt performance; it does not prove the feature has no predictive information.

Use a reproducible sample of up to 10,000 **Q1 tuning rows**, with five shuffles per feature. No model is refitted and no test/OOT data is used here. Error bars show variability across shuffles, not confidence intervals. Correlated inputs such as temperature summaries and trip lags can substitute for each other, so individual importance may understate their collective role.

We also shuffle related feature groups together to assess their joint importance. These are diagnostic associations, not causal effects. Weather importance refers to **same-day observed weather summaries**, including hourly coverage counts.

In [ ]:
from sklearn.inspection import permutation_importance

importance_X = X_tune.sample(n=min(10_000, len(X_tune)), random_state=RANDOM_STATE)
importance_y = y_tune.loc[importance_X.index]
importance_baseline_mae = mean_absolute_error(importance_y, best_model.predict(importance_X))
permutation = permutation_importance(
    best_model, importance_X, importance_y,
    scoring="neg_mean_absolute_error", n_repeats=5,
    random_state=RANDOM_STATE, n_jobs=1,
)
feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "MAE increase": permutation.importances_mean,
    "shuffle std": permutation.importances_std,
}).sort_values("MAE increase", ascending=False).reset_index(drop=True)
print(f"Importance sample: {len(importance_X):,} Q1 rows; baseline MAE: {importance_baseline_mae:.4f}")
display(feature_importance)

feature_groups = {
    "Trip history": [c for c in feature_columns if c.startswith(("fe_lag_", "fe_rolling_"))],
    "Weather (same day)": [c for c in numeric_columns if not c.startswith("fe_")],
    "Calendar": [c for c in feature_columns
                 if c in ("fe_day_of_week", "fe_month") or c.startswith("fe_is_")],
    "Station identity": ["station_id"],
}
grouped_columns = [c for columns in feature_groups.values() for c in columns]
missing_columns = sorted(set(feature_columns) - set(grouped_columns))
extra_columns = sorted(set(grouped_columns) - set(feature_columns))
duplicate_columns = sorted({c for c in grouped_columns if grouped_columns.count(c) > 1})
if missing_columns or extra_columns or duplicate_columns:
    raise ValueError(f"Feature grouping mismatch: missing={missing_columns}, "
                     f"extra={extra_columns}, duplicates={duplicate_columns}")
rng = np.random.default_rng(RANDOM_STATE)
group_records = []
for group, columns in feature_groups.items():
    increases = []
    for _ in range(5):
        shuffled = importance_X.copy()
        order = rng.permutation(len(shuffled))
        # One common permutation preserves relationships within the feature group.
        for column in columns:
            shuffled[column] = importance_X[column].to_numpy()[order]
        increases.append(mean_absolute_error(importance_y, best_model.predict(shuffled)) - importance_baseline_mae)
    group_records.append({"group": group, "MAE increase": np.mean(increases), "shuffle std": np.std(increases)})
group_importance = pd.DataFrame(group_records).sort_values("MAE increase", ascending=False)
display(group_importance)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
top = feature_importance.head(12).sort_values("MAE increase")
axes[0].barh(top["feature"], top["MAE increase"], xerr=top["shuffle std"], capsize=3)
axes[0].axvline(0, color="gray", linewidth=1)
axes[0].set(title="Top individual features", xlabel="Increase in MAE after shuffling")
groups = group_importance.sort_values("MAE increase")
axes[1].barh(groups["group"], groups["MAE increase"], xerr=groups["shuffle std"], capsize=3)
axes[1].axvline(0, color="gray", linewidth=1)
axes[1].set(title="Feature groups shuffled together", xlabel="Increase in MAE after shuffling")
plt.show()


## Evaluate MAE on test and both out-of-time periods

The seven-day lag is a simple demand baseline. Training scores are descriptive; tuning scores were used for selection. Q2, Q3 and Q4 scores are held-out results. RMSE and R² are included as secondary diagnostics, while **MAE remains the selection metric**.

In [ ]:
records = []
predictions = {}
for name, frame in splits.items():
    X, y = frame[feature_columns], frame["label"]
    model_predictions = {
        "Seven-day lag": frame["fe_lag_7"].to_numpy(),
        "Linear regression": ordinary_model.predict(X),
        "Tuned Ridge": best_model.predict(X),
    }
    predictions[name] = frame[KEYS + ["label"]].copy()
    for model_name, predicted in model_predictions.items():
        predictions[name][model_name] = predicted
        records.append({
            "split": name, "model": model_name, "rows": len(frame),
            "MAE": mean_absolute_error(y, predicted),
            "RMSE": np.sqrt(mean_squared_error(y, predicted)),
            "R2": r2_score(y, predicted),
            "negative_prediction_pct": 100 * np.mean(predicted < 0),
        })
metrics = pd.DataFrame(records)
display(metrics.pivot(index="split", columns="model", values="MAE").reindex(periods.keys()))
display(metrics.loc[metrics["model"].eq("Tuned Ridge")].set_index("split"))

In [ ]:
held_out_names = ["Test 2017 Q2", "OOT 1: 2017 Q3", "OOT 2: 2017 Q4"]
fig, axes = plt.subplots(2, 1, figsize=(13, 9), constrained_layout=True)
held_out_mae = metrics[metrics["split"].isin(held_out_names)].pivot(index="split", columns="model", values="MAE")
held_out_mae.reindex(held_out_names).plot.bar(ax=axes[0], rot=0)
axes[0].set(title="Held-out MAE by period", ylabel="MAE (trips per station/day)", xlabel="")

held_out = pd.concat([predictions[name] for name in held_out_names], ignore_index=True)
daily_totals = held_out.groupby("snapshot_date")[["label", "Tuned Ridge"]].sum()
daily_totals.rename(columns={"label": "Actual", "Tuned Ridge": "Predicted"}).plot(ax=axes[1], linewidth=1)
axes[1].set(title="Held-out demand summed across active stations", ylabel="Daily trip starts", xlabel="Date")
plt.show()

## Read the results

- Compare tuned Ridge with ordinary linear regression and the seven-day-lag baseline using MAE.
- Compare Q2 with Q3 and Q4 to inspect seasonal changes and performance drift.
- Rerunning search after inspecting held-out results would turn those periods into additional tuning data; preserve them for evaluation.
- `best_model`, `ordinary_model`, `metrics`, `search_results`, and per-split `predictions` remain available for further analysis. No existing gold data is changed.

## Compare with models that do not use weather

Keep trip history, station identity and calendar features, and remove all same-day weather inputs. Retune Ridge with the **same 20 sampled parameter combinations**, the same 2016 training rows and the same Q1 validation rows. Fit both the selected Ridge and an ordinary linear regression baseline on 2016 only.

This is a separate feature-set experiment; the weather and no-weather variants are both evaluated below. Q2, Q3 and Q4 are reported for comparison, with no parameter selection on those periods. Because their results have already been inspected, treat this as an exploratory comparison rather than a new untouched final evaluation. Trip inputs use history through D-1. The weather variant uses target-day observations; the no-weather variant avoids that availability limitation.

In [ ]:
weather_columns = [c for c in numeric_columns if not c.startswith("fe_")]
no_weather_columns = [c for c in feature_columns if c not in weather_columns]
no_weather_numeric = [c for c in numeric_columns if c not in weather_columns]
assert weather_columns and not any(c.endswith("_7d") for c in weather_columns)
assert not set(weather_columns) & set(no_weather_columns)
print("Removed weather features:", weather_columns)
print("Retained features:", no_weather_columns)

# Reuse the same preprocessing steps, restricting them to retained columns.
no_weather_preprocessor = ColumnTransformer([
    (name, clone(transformer), [c for c in columns if c in no_weather_columns])
    for name, transformer, columns in preprocessor.transformers
], sparse_threshold=1.0)
no_weather_pipeline = Pipeline([
    ("preprocess", no_weather_preprocessor),
    ("model", Ridge(solver="lsqr", tol=1e-4)),
])
no_weather_search = clone(search).set_params(estimator=no_weather_pipeline)
no_weather_search.fit(X_search[no_weather_columns], y_search)
assert no_weather_search.cv_results_["params"] == search.cv_results_["params"]
print("Best no-weather parameters:", no_weather_search.best_params_)
print(f"Best no-weather tuning MAE: {-no_weather_search.best_score_:.4f}")

no_weather_model = clone(no_weather_pipeline).set_params(**no_weather_search.best_params_)
no_weather_model.fit(X_train[no_weather_columns], y_train)
no_weather_ordinary_model = Pipeline([
    ("preprocess", clone(no_weather_preprocessor)), ("model", LinearRegression()),
])
no_weather_ordinary_model.fit(X_train[no_weather_columns], y_train)


In [ ]:
no_weather_records = []
no_weather_predictions = {}
for split_name, frame in splits.items():
    X = frame[no_weather_columns]
    y = frame["label"]
    no_weather_predictions[split_name] = frame[KEYS + ["label"]].copy()
    for model_name, model in [
        ("Linear regression (no weather)", no_weather_ordinary_model),
        ("Tuned Ridge (no weather)", no_weather_model),
    ]:
        predicted = model.predict(X)
        no_weather_predictions[split_name][model_name] = predicted
        no_weather_records.append({
            "split": split_name, "model": model_name, "rows": len(frame),
            "MAE": mean_absolute_error(y, predicted),
            "RMSE": np.sqrt(mean_squared_error(y, predicted)),
            "R2": r2_score(y, predicted),
            "negative_prediction_pct": 100 * np.mean(predicted < 0),
        })
no_weather_metrics = pd.DataFrame(no_weather_records)
comparison_metrics = pd.concat([
    metrics.assign(model=metrics["model"].replace({
        "Linear regression": "Linear regression (with weather)",
        "Tuned Ridge": "Tuned Ridge (with weather)",
    })), no_weather_metrics,
], ignore_index=True)
mae_comparison = comparison_metrics.pivot(index="split", columns="model", values="MAE").reindex(periods.keys())
display(mae_comparison)
ridge_weather_comparison = mae_comparison[["Tuned Ridge (with weather)", "Tuned Ridge (no weather)"]].copy()
ridge_weather_comparison["MAE increase without weather"] = (
    ridge_weather_comparison["Tuned Ridge (no weather)"] - ridge_weather_comparison["Tuned Ridge (with weather)"]
)
ridge_weather_comparison["MAE increase without weather (%)"] = (
    100 * ridge_weather_comparison["MAE increase without weather"] / ridge_weather_comparison["Tuned Ridge (with weather)"]
)
print("Positive differences mean weather improved MAE; negative differences favor no weather.")
display(ridge_weather_comparison)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
mae_comparison.loc[held_out_names, [
    "Seven-day lag", "Linear regression (no weather)",
    "Tuned Ridge (no weather)", "Tuned Ridge (with weather)",
]].plot.bar(ax=ax, rot=0)
ax.set(title="Weather versus no-weather models", ylabel="MAE (trips per station/day)", xlabel="")
ax.legend(title="Model", fontsize=9)
plt.show()
